# Comprehensive Benchmark of Ontology Semantic Similarity & Graph Distance Metrics

This notebook conducts a **rigorous head-to-head performance comparison** across all major ontology-based distance and semantic similarity metrics to identify the optimal scoring framework for downstream metadata concordance analysis (HAMLET LLM-agent vs MLMarker).

---

### Evaluated Distance & Semantic Similarity Models:

1. **Lin Information Content (IC) Semantic Similarity & Distance**:
   $$S_{\text{Lin}}(t_1, t_2) = \frac{2 \times \text{IC}(\text{MICA}(t_1, t_2))}{\text{IC}(t_1) + \text{IC}(t_2)} \in [0, 1], \quad d_{\text{Lin}} = 1 - S_{\text{Lin}}$$
2. **Wu & Palmer (WUP) Depth-Based Similarity & Distance**:
   $$S_{\text{WUP-depth}}(t_1, t_2) = \frac{2 \times \text{depth}(\text{LCS}(t_1, t_2))}{\text{depth}(t_1) + \text{depth}(t_2)} \in [0, 1], \quad d_{\text{WUP-depth}} = 1 - S_{\text{WUP-depth}}$$
3. **Wu & Palmer (WUP) Information Content-Based Similarity**:
   $$S_{\text{WUP-IC}}(t_1, t_2) = \frac{2 \times \text{IC}(\text{MICA}(t_1, t_2))}{\text{IC}(t_1) + \text{IC}(t_2)} = S_{\text{Lin}}(t_1, t_2)$$
4. **Information Content-Weighted Jaccard Similarity $J_{\text{IC}}$**:
   $$J_{\text{IC}}(t_1, t_2) = \frac{\sum_{a \in \text{Anc}(t_1) \cap \text{Anc}(t_2)} \text{IC}(a)}{\sum_{u \in \text{Anc}(t_1) \cup \text{Anc}(t_2)} \text{IC}(u)} \in [0, 1], \quad d_{\text{Jaccard-IC}} = 1 - J_{\text{IC}}$$
5. **Standard Filtered Ancestor Jaccard Similarity $J_{\text{anc}}$ (OAKlib)**:
   $$J_{\text{anc}}(t_1, t_2) = \frac{|\text{Anc}(t_1) \cap \text{Anc}(t_2)|}{|\text{Anc}(t_1) \cup \text{Anc}(t_2)|} \in [0, 1]$$
6. **Normalized Resnik Similarity**:
   $$S_{\text{Resnik-norm}}(t_1, t_2) = \frac{\text{IC}(\text{MICA}(t_1, t_2))}{\max(\text{IC}(t_1), \text{IC}(t_2))} \in [0, 1]$$
7. **1D Shortest Graph Path Distance $d_{\text{graph}}$ & Agreement $w_{\text{graph}}$** (`Try_distance.ipynb`):
   $$w_{\text{graph}}(A, B) = \frac{1}{1 + d_{\text{graph}}(A, B)}$$
8. **2D Metric MDS Spatial Distance $d_{\text{2D}}$ & Agreement $w_{\text{2D}}$**:
   $$w_{\text{2D}}(A, B) = \frac{1}{1 + d_{\text{2D}}(A, B)}$$
9. **Lin $k$-NN Similarity Graph Distance $d_{\text{Lin-Graph}}$**:
   - Shortest path distance through the $k=8$ nearest semantic neighbors network.

In [3]:
# 1. Import Dependencies, Configure Aesthetics and Load Dataset
import os
import re
import math
import warnings
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import cdist
from sklearn.manifold import MDS

# OAKlib imports
import oaklib
from oaklib.implementations.simpleobo.simple_obo_implementation import SimpleOboImplementation
from oaklib.resource import OntologyResource

warnings.filterwarnings('ignore')

# Configure publication aesthetics
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['grid.color'] = '#eeeeee'
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['figure.dpi'] = 120

# Workspace paths
PATH_COMBINED = r"C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\agentic-metadata\run_metadata_combined.tsv"
PATH_MLM = r"C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\agentic-metadata\run_meta_mlmarker.tsv"
PATH_AGENT = r"C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\agentic-metadata\agent_metadata.tsv"
PATH_ONTOLOGY_DIR = r"C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\ontologies"
PATH_OUTPUT = r"C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\output\Try_distance"

os.makedirs(PATH_OUTPUT, exist_ok=True)
os.makedirs(PATH_ONTOLOGY_DIR, exist_ok=True)

# Load datasets
df_combined = pd.read_csv(PATH_COMBINED, sep="\t", low_memory=False)
df_mlm = pd.read_csv(PATH_MLM, sep="\t", low_memory=False).drop_duplicates(subset=['pxd', 'run'], keep='first')
df_agent = pd.read_csv(PATH_AGENT, sep="\t", low_memory=False)

# Merge MLMarker prediction onto combined metadata
df_merged = pd.merge(
    df_combined,
    df_mlm[['pxd', 'run', 'tissue', 'confidence']],
    on=['pxd', 'run'],
    how='left',
    suffixes=('', '_mlm')
)

# Filter overlapping runs
mask_both = df_merged['tissue_evidence'].str.contains(r'(?=.*agent)(?=.*MLM)', regex=True, na=False)
df_filtered = df_merged[mask_both].copy()
df_filtered['is_agree'] = ~df_filtered['tissue_evidence'].str.contains('-MLM', na=False)
df_filtered['tissue_agent_norm'] = df_filtered['tissue'].str.lower().str.strip()
df_filtered['tissue_mlm_norm'] = df_filtered['tissue_mlm'].str.lower().str.strip()

print(f"Environment ready. OAKlib version: {oaklib.__version__}, NetworkX: {nx.__version__}, Pandas: {pd.__version__}")
print(f"Total runs evaluated (HAMLET & MLMarker): {df_filtered.shape[0]:,}")
print(f"  - Raw exact matches: {df_filtered['is_agree'].sum():,} ({df_filtered['is_agree'].mean()*100:.2f}%)")
print(f"  - Raw non-exact runs: {(~df_filtered['is_agree']).sum():,} ({(~df_filtered['is_agree']).mean()*100:.2f}%)")

Environment ready. OAKlib version: 0.7.4, NetworkX: 3.6.1, Pandas: 2.3.3
Total runs evaluated (HAMLET & MLMarker): 4,676
  - Raw exact matches: 3,600 (76.99%)
  - Raw non-exact runs: 1,076 (23.01%)


| Blocked Concept ID | Label | Why it is Blocked |
| --- | --- | --- |
| UBERON:0000062 | organ | **High-degree generic hubs:** Every organ/tissue connects here. Leaving them open causes all unrelated human organs to falsely appear 2 hops away from each other. |
| UBERON:0000064 | organ part | **High-degree generic hubs:** Every organ/tissue connects here. Leaving them open causes all unrelated human organs to falsely appear 2 hops away from each other. |
| UBERON:0000061 | anatomical structure | **High-degree generic hubs:** Every organ/tissue connects here. Leaving them open causes all unrelated human organs to falsely appear 2 hops away from each other. |
| UBERON:0001062 | anatomical entity | **High-degree generic hubs:** Every organ/tissue connects here. Leaving them open causes all unrelated human organs to falsely appear 2 hops away from each other. |
| UBERON:0000479 | tissue | **High-degree generic hubs:** Every organ/tissue connects here. Leaving them open causes all unrelated human organs to falsely appear 2 hops away from each other. |
| UBERON:0004120 | mesoderm-derived structure | **Embryological shortcuts:** Almost all internal organs (heart, kidney, muscle, blood, ovary) derive from mesoderm. Leaving these open connects kidney and heart in 2 hops based on early embryonic origin rather than physiological function. |
| UBERON:0004121 | ectoderm-derived structure | **Embryological shortcuts:** Almost all internal organs (heart, kidney, muscle, blood, ovary) derive from mesoderm. Leaving these open connects kidney and heart in 2 hops based on early embryonic origin rather than physiological function. |
| UBERON:0004119 | endoderm-derived structure | **Embryological shortcuts:** Almost all internal organs (heart, kidney, muscle, blood, ovary) derive from mesoderm. Leaving these open connects kidney and heart in 2 hops based on early embryonic origin rather than physiological function. |
| UBERON:0000078 | portion of organism substance | **Fluid hub:** All body fluids (saliva, urine, CSF, semen, blood plasma) connect here, causing all fluids to short-circuit to each other rather than to their source organs. |
| BFO:0000002 | continuant | **Top-level philosophical roots:** Universal ancestors at the root of all biomedical ontologies that carry zero anatomical specificity. |
| BFO:0000004 | independent continuant | **Top-level philosophical roots:** Universal ancestors at the root of all biomedical ontologies that carry zero anatomical specificity. |
| BFO:0000040 | material entity | **Top-level philosophical roots:** Universal ancestors at the root of all biomedical ontologies that carry zero anatomical specificity. |

In [4]:
# 2. Build Strict Anatomical & Cellular Knowledge Graph (UBERON + CL)
import os
import re
import math
import numpy as np
import networkx as nx

uberon_obo_path = os.path.join(PATH_ONTOLOGY_DIR, "uberon-basic.obo")
cl_obo_path = os.path.join(PATH_ONTOLOGY_DIR, "cl.obo")

# Could adjust weights or includ more relations based on domain knowledge 
RELATION_WEIGHTS = {
    'is_a': 1.0,
    'part_of': 1.0, 'BFO:0000050': 1.0,
    'has_part': 1.0, 'BFO:0000051': 1.0,
    'subdivision_of': 1.0,
    'located_in': 0.8, 'RO:0001025': 0.8,
    'location_of': 0.8, 'RO:0001015': 0.8,
    'composed_primarily_of': 1.1, 'RO:0002473': 1.1,
    'secreted_by': 1.2,
    'produces': 1.2, 'RO:0003000': 1.2,
    'produced_by': 1.2, 'RO:0003001': 1.2,
    'connected_to': 1.3, 'RO:0002150': 1.3,
    'continuous_with': 1.3,
    'develops_from': 1.5, 'RO:0002202': 1.5,
    'develops_in': 1.5, 'RO:0002203': 1.5,
    'sexually_homologous_to': 1.5,
    'transformation_of': 1.5, 'RO:0002494': 1.5
}

BLOCKED_PREFIXES = {'NCBITaxon', 'CARO', 'PATO', 'IAO', 'STATO', 'SO', 'PR', 'http', 'https', 'CHEBI'}
BLOCKED_NODES = {
    'UBERON:0000062', 'UBERON:0000064', 'UBERON:0000061', 'UBERON:0000465', 'UBERON:0001062',
    'UBERON:0000475', 'UBERON:0000479', 'UBERON:0000478', 'UBERON:0015212', 'UBERON:0004121',
    'UBERON:0004120', 'UBERON:0004119', 'UBERON:0000078', 'BFO:0000002', 'BFO:0000004', 'BFO:0000040',
}

def build_knowledge_graph(use_blocked_nodes=True):
    G_dir = nx.DiGraph()
    G_wei = nx.Graph()
    n2i = {}
    i2n = {}

    for obo_file in [uberon_obo_path, cl_obo_path]:
        with open(obo_file, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        
        term_pattern = re.compile(r'\[Term\](.*?)(?=\n\[|\Z)', re.DOTALL)
        for block in term_pattern.findall(content):
            lines = block.strip().split('\n')
            node_id, name = None, None
            synonyms, parents = [], []
            is_obsolete = False
            
            for line in lines:
                line = line.strip()
                if line.startswith('id:'):
                    node_id = line[3:].strip()
                elif line.startswith('name:'):
                    name = line[5:].strip()
                elif line.startswith('synonym:'):
                    m = re.search(r'"([^"]+)"', line)
                    if m:
                        synonyms.append(m.group(1))
                elif line.startswith('is_a:'):
                    pid = line[5:].split('!')[0].strip().split('{')[0].strip()
                    parents.append((pid, 'is_a'))
                elif line.startswith('relationship:'):
                    parts = line[13:].strip().split()
                    if len(parts) >= 2:
                        rel_type = parts[0]
                        pid = parts[1].split('!')[0].strip().split('{')[0].strip()
                        parents.append((pid, rel_type))
                elif line.startswith('is_obsolete:') and 'true' in line.lower():
                    is_obsolete = True
            
            if node_id and name and not is_obsolete:
                prefix = node_id.split(':')[0] if ':' in node_id else ''
                if prefix in BLOCKED_PREFIXES:
                    continue
                if use_blocked_nodes and node_id in BLOCKED_NODES:
                    continue
                
                G_dir.add_node(node_id, name=name)
                G_wei.add_node(node_id, name=name)
                i2n[node_id] = name
                n2i[name.lower().strip()] = node_id
                for s in synonyms:
                    n2i[s.lower().strip()] = node_id
                
                for pid, rel in parents:
                    p_prefix = pid.split(':')[0] if ':' in pid else ''
                    if p_prefix in BLOCKED_PREFIXES or 'taxon' in rel.lower():
                        continue
                    if use_blocked_nodes and pid in BLOCKED_NODES:
                        continue
                    if rel in RELATION_WEIGHTS or rel.startswith('is_a'):
                        base_w = RELATION_WEIGHTS.get(rel, 1.0)
                        G_dir.add_edge(node_id, pid, relation=rel, weight=base_w)
                        G_wei.add_edge(node_id, pid, relation=rel, weight=base_w)

    return G_dir, G_wei, n2i, i2n

G_directed_b, G_weighted_b, name_to_id_b, id_to_name_b = build_knowledge_graph(True)
G_directed_u, _, _, _ = build_knowledge_graph(False)

G_directed = G_directed_b
G_weighted = G_weighted_b
name_to_id = name_to_id_b
id_to_name = id_to_name_b

roots = [n for n, d in G_directed.out_degree() if d == 0]
depth_dict = {}
rev_G = G_directed.reverse()
for root in roots:
    lengths = nx.single_source_shortest_path_length(rev_G, root)
    for n, d in lengths.items():
        depth_dict[n] = max(depth_dict.get(n, 1), d + 1)

for u, v, data in G_weighted.edges(data=True):
    base_w = data.get('weight', 1.0)
    du = depth_dict.get(u, 1)
    dv = depth_dict.get(v, 1)
    depth_factor = 2.0 / (np.sqrt(du) + np.sqrt(dv))
    data['distance'] = round(float(base_w * depth_factor), 4)

print(f"Knowledge Graph built (Blocked): {len(G_weighted_b):,} nodes, {G_weighted_b.number_of_edges():,} edges.")
print(f"Knowledge Graph built (Unblocked): {len(G_directed_u):,} nodes, {G_directed_u.number_of_edges():,} edges.")

def compute_ti_map(G_dir):
    G_rev = G_dir.reverse()
    while not nx.is_directed_acyclic_graph(G_rev):
        try:
            cycle = nx.find_cycle(G_rev)
            G_rev.remove_edge(cycle[-1][0], cycle[-1][1])
        except nx.NetworkXNoCycle:
            break
            
    top_order = list(nx.topological_sort(G_rev))
    ti_map = {}
    roots_rev = [n for n, d in G_rev.in_degree() if d == 0]
    for r in roots_rev:
        ti_map[r] = 0.0
        
    for node in top_order:
        if node in roots_rev:
            continue
        parents = list(G_rev.predecessors(node))
        val = 0.0
        for p in parents:
            children_p_count = G_rev.out_degree(p)
            ti_p = ti_map.get(p, 0.0)
            if children_p_count > 0:
                val += (ti_p + math.log2(children_p_count))
        ti_map[node] = val
    return ti_map

_ti_map_b = compute_ti_map(G_directed_b)
_ti_map_u = compute_ti_map(G_directed_u)
print(f"Computed TI maps for Blocked and Unblocked graphs.")



Knowledge Graph built (Blocked): 26,792 nodes, 51,955 edges.
Knowledge Graph built (Unblocked): 26,808 nodes, 54,133 edges.
Computed TI maps for Blocked and Unblocked graphs.


In [5]:
# 3. Canonical Entity Resolver & Comprehensive Multi-Metric Engine
term_to_uberon = {
    'brain': 'UBERON:0000955', 'cerebral cortex': 'UBERON:0000956', 'heart': 'UBERON:0000948',
    'liver': 'UBERON:0002107', 'lung': 'UBERON:0002048', 'kidney': 'UBERON:0002113',
    'cortex of kidney': 'UBERON:0001225', 'testis': 'UBERON:0000473', 'ovary': 'UBERON:0000992',
    'prostate': 'UBERON:0002367', 'salivary gland': 'UBERON:0001044', 'saliva': 'UBERON:0001836',
    'placenta': 'UBERON:0001987', 'esophagus': 'UBERON:0001043', 'colon': 'UBERON:0001155',
    'small intestine': 'UBERON:0002108', 'duodenum': 'UBERON:0002114', 'stomach': 'UBERON:0000945',
    'tonsil': 'UBERON:0002372', 'skeletal muscle': 'UBERON:0001134', 'urinary bladder': 'UBERON:0001255',
    'thyroid': 'UBERON:0002046', 'thyroid gland': 'UBERON:0002046', 'endometrium': 'UBERON:0001295',
    'uterine endometrium': 'UBERON:0001295', 'oviduct': 'UBERON:0000993', 'fallopian tube': 'UBERON:0003889',
    'adrenal gland': 'UBERON:0002369', 'pituitary gland': 'UBERON:0000007', 'hypophysis': 'UBERON:0000007',
    'adipose tissue': 'UBERON:0001013', 'bone marrow': 'UBERON:0002371', 'blood': 'UBERON:0000178',
    'blood plasma': 'UBERON:0001969', 'blood serum': 'UBERON:0001977', 'seminal plasma': 'UBERON:0006530',
    'sperm': 'UBERON:0001968', 'urine': 'UBERON:0001088', 'cerebrospinal fluid': 'UBERON:0001359',
    'tear fluid': 'UBERON:0001835', 'pleural fluid': 'UBERON:0001407', 'peritoneal fluid': 'UBERON:0001270',
    'monocytes': 'CL:0000576', 'monocyte': 'CL:0000576', 'b-cells': 'CL:0000236',
    'b cell': 'CL:0000236', 'b cells': 'CL:0000236', 'pbmcs': 'CL:2000001', 'pbmc': 'CL:2000001',
    't cells': 'CL:0000084', 't cell': 'CL:0000084', 'neutrophil': 'CL:0000775',
    'stem cells': 'CL:0000034', 'stem cell': 'CL:0000034', 'nasal polyps': 'UBERON:0005086',
    'bronchoalveolar lavage fluid': 'UBERON:0006326', 'skin': 'UBERON:0002097', 'gallbladder': 'UBERON:0002110',
    'lymph node': 'UBERON:0000029', 'pancreas': 'UBERON:0001264', 'rectum': 'UBERON:0001052',
    'smooth muscle': 'UBERON:0001135', 'spleen': 'UBERON:0002106', 'vermiform appendix': 'UBERON:0001154',
}

def resolve_term(term):
    if not term or pd.isna(term):
        return None
    t = str(term).lower().strip()
    if t in term_to_uberon:
        return term_to_uberon[t]
    if t in name_to_id:
        return name_to_id[t]
    matches = [(k, v) for k, v in term_to_uberon.items() if k in t]
    if matches:
        best = max(matches, key=lambda x: len(x[0]))
        return best[1]
    return None

_path_cache = {}
def get_cached_path_and_dist(u, v):
    pair = (u, v)
    if pair in _path_cache:
        return _path_cache[pair]
    if u == v:
        res = (0.0, 0, [u])
        _path_cache[pair] = res
        return res
    if not (u in G_weighted and v in G_weighted and nx.has_path(G_weighted, u, v)):
        _path_cache[pair] = (np.inf, np.nan, [])
        return (np.inf, np.nan, [])
    path = nx.shortest_path(G_weighted, u, v, weight='distance')
    d_val = sum(G_weighted[path[i]][path[i+1]].get('distance', 1.0) for i in range(len(path) - 1))
    d_val = round(float(d_val), 4)
    res = (d_val, len(path) - 1, path)
    _path_cache[pair] = res
    _path_cache[(v, u)] = (d_val, len(path) - 1, list(reversed(path)))
    return res

VALID_ANCESTOR_PREFIXES = {'UBERON:', 'CL:', 'BFO:'}
_anc_cache_b = {}
def get_filtered_ancestors_b(curie):
    if curie not in _anc_cache_b:
        try:
            raw_anc = nx.descendants(G_directed_b, curie) | {curie}
            _anc_cache_b[curie] = {a for a in raw_anc if any(str(a).startswith(p) for p in VALID_ANCESTOR_PREFIXES)}
        except Exception:
            _anc_cache_b[curie] = {curie}
    return _anc_cache_b[curie]

_anc_cache_u = {}
def get_filtered_ancestors_u(curie):
    if curie not in _anc_cache_u:
        try:
            raw_anc = nx.descendants(G_directed_u, curie) | {curie}
            _anc_cache_u[curie] = {a for a in raw_anc if any(str(a).startswith(p) for p in VALID_ANCESTOR_PREFIXES)}
        except Exception:
            _anc_cache_u[curie] = {curie}
    return _anc_cache_u[curie]

_total_nodes = len(G_directed)
_ic_map = {}
for node in G_directed.nodes():
    try:
        desc_count = len(nx.descendants(rev_G, node)) + 1
        _ic_map[node] = -math.log2(desc_count / _total_nodes)
    except Exception:
        _ic_map[node] = 1.0

def compute_graphic_sim(id_a, id_b, ti_map, get_anc_func):
    anc_a = get_anc_func(id_a)
    anc_b = get_anc_func(id_b)
    inter = anc_a.intersection(anc_b)
    if not inter:
        return 0.0, 1.0, 'None'
    mica_id = max(inter, key=lambda a: ti_map.get(a, 0.0))
    mica_ti = ti_map.get(mica_id, 0.0)
    ti_a = ti_map.get(id_a, 1.0)
    ti_b = ti_map.get(id_b, 1.0)
    
    sim = (2.0 * mica_ti) / (ti_a + ti_b) if (ti_a + ti_b) > 0 else 0.0
    sim = round(min(1.0, max(0.0, float(sim))), 4)
    dist = round(1.0 - sim, 4)
    return sim, dist, mica_id

def compute_all_metrics_pairwise(id_a, id_b):
    if not id_a or not id_b:
        return {}
    if id_a == id_b:
        ic_val = _ic_map.get(id_a, 0.0)
        lbl = id_to_name.get(id_a, id_a)
        # Handle the case where both IDs are the same
        return {
            'graph_dist_1d': 0.0, 'graph_sim_1d': 1.0,
            'lin_sim': 1.0, 'lin_dist': 0.0,
            'wup_depth_sim': 1.0, 'wup_depth_dist': 0.0,
            'wup_ic_sim': 1.0, 'wup_ic_dist': 0.0,
            'jaccard_ic_sim': 1.0, 'jaccard_ic_dist': 0.0,
            'jaccard_anc_sim': 1.0, 'jaccard_anc_dist': 0.0,
            'resnik_norm_sim': 1.0,
            'graphic_sim_b': 1.0, 'graphic_sim_u': 1.0,
            'mica_id': id_a, 'mica_label': lbl, 'mica_ic': round(float(ic_val), 4)
        }
        
    d_1d, _, _ = get_cached_path_and_dist(id_a, id_b)
    w_1d = round(1.0 / (1.0 + d_1d), 4) if not np.isinf(d_1d) else 0.0
    
    anc_a = get_filtered_ancestors_b(id_a)
    anc_b = get_filtered_ancestors_b(id_b)
    inter = anc_a.intersection(anc_b)
    union = anc_a.union(anc_b)
    
    j_anc = round(float(len(inter) / len(union)), 4) if union else 0.0
    sum_inter_ic = sum(_ic_map.get(a, 0.0) for a in inter)
    sum_union_ic = sum(_ic_map.get(a, 0.0) for a in union)
    j_ic = round(float(sum_inter_ic / sum_union_ic), 4) if sum_union_ic > 0 else 0.0
    
    gs_b, gd_b, _ = compute_graphic_sim(id_a, id_b, _ti_map_b, get_filtered_ancestors_b)
    gs_u, gd_u, _ = compute_graphic_sim(id_a, id_b, _ti_map_u, get_filtered_ancestors_u)

    # Handle the case where there are no common ancestors
    if not inter:
        return {
            'graph_dist_1d': d_1d, 'graph_sim_1d': w_1d,
            'lin_sim': 0.0, 'lin_dist': 1.0,
            'wup_depth_sim': 0.0, 'wup_depth_dist': 1.0,
            'wup_ic_sim': 0.0, 'wup_ic_dist': 1.0,
            'jaccard_ic_sim': j_ic, 'jaccard_ic_dist': round(1.0 - j_ic, 4),
            'jaccard_anc_sim': j_anc, 'jaccard_anc_dist': round(1.0 - j_anc, 4),
            'resnik_norm_sim': 0.0,
            'graphic_sim_b': gs_b, 'graphic_sim_u': gs_u,
            'mica_id': 'None', 'mica_label': 'None', 'mica_ic': 0.0
        }
    
    mica_id = max(inter, key=lambda a: _ic_map.get(a, 0.0))
    mica_ic = _ic_map.get(mica_id, 0.0)
    ic_a = _ic_map.get(id_a, 1.0)
    ic_b = _ic_map.get(id_b, 1.0)
    
    lin_s = (2.0 * mica_ic) / (ic_a + ic_b) if (ic_a + ic_b) > 0 else 0.0
    lin_s = round(min(1.0, max(0.0, float(lin_s))), 4)
    lin_d = round(1.0 - lin_s, 4)
    
    lcs_id = max(inter, key=lambda a: depth_dict.get(a, 1))
    depth_lcs = depth_dict.get(lcs_id, 1)
    depth_a = depth_dict.get(id_a, 1)
    depth_b = depth_dict.get(id_b, 1)
    wup_d_s = (2.0 * depth_lcs) / (depth_a + depth_b) if (depth_a + depth_b) > 0 else 0.0
    wup_d_s = round(min(1.0, max(0.0, float(wup_d_s))), 4)
    wup_d_d = round(1.0 - wup_d_s, 4)
    
    max_ic = max(ic_a, ic_b)
    resnik_norm_s = round(float(mica_ic / max_ic), 4) if max_ic > 0 else 0.0
    resnik_norm_s = min(1.0, max(0.0, resnik_norm_s))
    
    return {
        'graph_dist_1d': d_1d, 'graph_sim_1d': w_1d,
        'lin_sim': lin_s, 'lin_dist': lin_d,
        'wup_depth_sim': wup_d_s, 'wup_depth_dist': wup_d_d,
        'wup_ic_sim': lin_s, 'wup_ic_dist': lin_d,
        'jaccard_ic_sim': j_ic, 'jaccard_ic_dist': round(1.0 - j_ic, 4),
        'jaccard_anc_sim': j_anc, 'jaccard_anc_dist': round(1.0 - j_anc, 4),
        'resnik_norm_sim': resnik_norm_s,
        'graphic_sim_b': gs_b, 'graphic_sim_u': gs_u,
        'mica_id': mica_id, 'mica_label': id_to_name.get(mica_id, mica_id), 'mica_ic': round(float(mica_ic), 4)
    }

print(f"Multi-metric engine configured for {len(_ic_map):,} concepts.")


Multi-metric engine configured for 26,792 concepts.


In [6]:
# 4. 2D Metric MDS Embedding and Lin k-NN Graph Setup
corpus_terms = set(term_to_uberon.keys())
for t in df_filtered['tissue'].dropna().unique():
    for p in str(t).split(';'):
        if p.strip():
            corpus_terms.add(p.strip().lower())
for t in df_filtered['tissue_mlm'].dropna().unique():
    if str(t).strip():
        corpus_terms.add(str(t).strip().lower())

term_to_node = {}
for t in corpus_terms:
    nid = resolve_term(t)
    if nid and nid in G_weighted:
        term_to_node[t] = nid

unique_corpus_nodes = sorted(list(set(term_to_node.values())))
n_nodes = len(unique_corpus_nodes)
node_to_idx = {nid: i for i, nid in enumerate(unique_corpus_nodes)}

# 1. 1D Graph Distance Matrix
D_graph_full = np.zeros((n_nodes, n_nodes))
for i, u in enumerate(unique_corpus_nodes):
    for j, v in enumerate(unique_corpus_nodes):
        if i == j:
            D_graph_full[i, j] = 0.0
        elif j > i:
            d_val, _, _ = get_cached_path_and_dist(u, v)
            D_graph_full[i, j] = d_val if not np.isinf(d_val) else 4.0
            D_graph_full[j, i] = D_graph_full[i, j]

# 2. 2D Metric MDS Coordinates
mds_model = MDS(n_components=2, dissimilarity='precomputed', random_state=42, n_init=15, max_iter=600)
coords_2d = mds_model.fit_transform(D_graph_full)
node_coords_2d = {nid: coords_2d[i] for i, nid in enumerate(unique_corpus_nodes)}
term_coords_2d = {t: node_coords_2d[nid] for t, nid in term_to_node.items()}

# 3. Lin k-NN Graph Model (k=8)
G_sim_lin = nx.Graph()
for nid in unique_corpus_nodes:
    G_sim_lin.add_node(nid, name=id_to_name.get(nid, nid))

for i, u in enumerate(unique_corpus_nodes):
    sims = []
    for j, v in enumerate(unique_corpus_nodes):
        if i != j:
            scores = compute_all_metrics_pairwise(u, v)
            sims.append((j, scores['lin_sim']))
    sims.sort(key=lambda x: x[1], reverse=True)
    for j, s in sims[:8]:
        v = unique_corpus_nodes[j]
        d_edge = round(float(1.0 - s), 4)
        G_sim_lin.add_edge(u, v, weight=d_edge, similarity=s)

print(f"2D MDS and Lin k-NN Graph initialized for {n_nodes} nodes.")

2D MDS and Lin k-NN Graph initialized for 70 nodes.


In [7]:
# 5. 8-Panel Pairwise Comparison Heatmap Across All 26 MLMarker Tissues
mlm_tissues = sorted([t.lower().strip() for t in df_mlm['tissue'].dropna().unique()])
n_mlm = len(mlm_tissues)

m_graph_1d = np.zeros((n_mlm, n_mlm))
m_spatial_2d = np.zeros((n_mlm, n_mlm))
m_lin_sim = np.zeros((n_mlm, n_mlm))
m_wup_depth = np.zeros((n_mlm, n_mlm))
m_jaccard_ic = np.zeros((n_mlm, n_mlm))
m_resnik_norm = np.zeros((n_mlm, n_mlm))
m_graphic_b = np.zeros((n_mlm, n_mlm))
m_graphic_u = np.zeros((n_mlm, n_mlm))

for i, t1 in enumerate(mlm_tissues):
    id1 = resolve_term(t1)
    p1 = term_coords_2d.get(t1)
    for j, t2 in enumerate(mlm_tissues):
        if i == j or t1 == t2:
            m_graph_1d[i, j] = 0.0
            m_spatial_2d[i, j] = 0.0
            m_lin_sim[i, j] = 1.0
            m_wup_depth[i, j] = 1.0
            m_jaccard_ic[i, j] = 1.0
            m_resnik_norm[i, j] = 1.0
            m_graphic_b[i, j] = 1.0
            m_graphic_u[i, j] = 1.0
            continue
            
        id2 = resolve_term(t2)
        p2 = term_coords_2d.get(t2)
        
        if id1 and id2:
            scores = compute_all_metrics_pairwise(id1, id2)
            m_graph_1d[i, j] = scores['graph_dist_1d'] if not np.isinf(scores['graph_dist_1d']) else np.nan
            m_lin_sim[i, j] = scores['lin_sim']
            m_wup_depth[i, j] = scores['wup_depth_sim']
            m_jaccard_ic[i, j] = scores['jaccard_ic_sim']
            m_resnik_norm[i, j] = scores['resnik_norm_sim']
            m_graphic_b[i, j] = scores['graphic_sim_b']
            m_graphic_u[i, j] = scores['graphic_sim_u']
        else:
            m_graph_1d[i, j] = np.nan
            m_lin_sim[i, j] = np.nan
            m_wup_depth[i, j] = np.nan
            m_jaccard_ic[i, j] = np.nan
            m_resnik_norm[i, j] = np.nan
            m_graphic_b[i, j] = np.nan
            m_graphic_u[i, j] = np.nan
            
        if p1 is not None and p2 is not None:
            m_spatial_2d[i, j] = round(float(np.linalg.norm(p1 - p2)), 4)
        else:
            m_spatial_2d[i, j] = np.nan

tissue_labels = [t.capitalize() for t in mlm_tissues]
df_m_graph_1d = pd.DataFrame(m_graph_1d, index=tissue_labels, columns=tissue_labels)
df_m_spatial_2d = pd.DataFrame(m_spatial_2d, index=tissue_labels, columns=tissue_labels)
df_m_lin_sim = pd.DataFrame(m_lin_sim, index=tissue_labels, columns=tissue_labels)
df_m_wup_depth = pd.DataFrame(m_wup_depth, index=tissue_labels, columns=tissue_labels)
df_m_jaccard_ic = pd.DataFrame(m_jaccard_ic, index=tissue_labels, columns=tissue_labels)
df_m_resnik_norm = pd.DataFrame(m_resnik_norm, index=tissue_labels, columns=tissue_labels)
df_m_graphic_b = pd.DataFrame(m_graphic_b, index=tissue_labels, columns=tissue_labels)
df_m_graphic_u = pd.DataFrame(m_graphic_u, index=tissue_labels, columns=tissue_labels)

# 8-Panel Heatmap Figure
fig, axes = plt.subplots(2, 4, figsize=(40, 18))

panels = [
    (df_m_graph_1d, "A. 1D Graph Shortest Distance $d_{\\text{graph}}$", "viridis_r", axes[0, 0]),
    (df_m_spatial_2d, "B. 2D Spatial Distance $d_{\\text{2D}}$", "viridis_r", axes[0, 1]),
    (df_m_lin_sim, "C. Lin Semantic Similarity $S_{\\text{Lin}}$", "viridis", axes[0, 2]),
    (df_m_wup_depth, "D. Wu & Palmer Depth Similarity $\\text{WUP}_{\\text{depth}}$", "viridis", axes[1, 0]),
    (df_m_jaccard_ic, "E. Jaccard IC Similarity $J_{\\text{IC}}$", "viridis", axes[1, 1]),
    (df_m_resnik_norm, "F. Normalized Resnik Similarity $\\text{Resnik}_{\\text{norm}}$", "viridis", axes[1, 2]),
    (df_m_graphic_b, "G. GraphIC Sim (Blocked) $S_{\\text{GraphIC,b}}$", "viridis", axes[0, 3]),
    (df_m_graphic_u, "H. GraphIC Sim (Unblocked) $S_{\\text{GraphIC,u}}$", "viridis", axes[1, 3]),
]

for df_p, title_p, cmap_p, ax_p in panels:
    sns.heatmap(df_p, cmap=cmap_p, annot=False, cbar_kws={'shrink': 0.8}, linewidths=0.3, linecolor='#f0f0f0', ax=ax_p)
    ax_p.set_title(title_p, fontsize=12, weight='bold', pad=10)
    ax_p.set_xticklabels(ax_p.get_xticklabels(), rotation=45, ha='right', fontsize=7.5, weight='bold')
    ax_p.set_yticklabels(ax_p.get_yticklabels(), rotation=0, fontsize=7.5, weight='bold')

plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, "pairwise_all_metrics_heatmaps.png"), dpi=300, bbox_inches='tight')
plt.close()

print("8-Panel Multi-Metric Heatmap saved to Output/pairwise_all_metrics_heatmaps.png")


8-Panel Multi-Metric Heatmap saved to Output/pairwise_all_metrics_heatmaps.png


In [8]:
# 6. Benchmark Biological Case Studies Across All Evaluated Metrics
case_study_pairs = [
    ('saliva', 'salivary gland', 'Secretome / Fluid'),
    ('kidney', 'cortex of kidney', 'Organ Subpart'),
    ('blood plasma', 'blood', 'Fluid Fraction'),
    ('brain', 'cerebrospinal fluid', 'Fluid / CNS System'),
    ('brain', 'cerebral cortex', 'Organ Regional Part'),
    ('monocytes', 'blood', 'Cellular Lineage / Localization'),
    ('prostate', 'seminal plasma', 'Secretome / Fluid'),
    ('small intestine', 'duodenum', 'Organ Subdivision'),
    ('colon', 'small intestine', 'Digestive Neighbors'),
    ('brain', 'heart', 'Distant Negative Control'),
    ('skin', 'ovary', 'True Biological Conflict'),
]

case_results = []
for t1, t2, rel_desc in case_study_pairs:
    id1 = resolve_term(t1)
    id2 = resolve_term(t2)
    p1 = term_coords_2d.get(t1)
    p2 = term_coords_2d.get(t2)
    
    scores = compute_all_metrics_pairwise(id1, id2) if id1 and id2 else {}
    d2 = float(np.linalg.norm(p1 - p2)) if p1 is not None and p2 is not None else np.nan
    w2 = round(1.0 / (1.0 + d2), 4) if not np.isnan(d2) else np.nan
    
    # Lin Graph Dist
    if id1 and id2 and id1 in G_sim_lin and id2 in G_sim_lin and nx.has_path(G_sim_lin, id1, id2):
        p_sim = nx.shortest_path(G_sim_lin, id1, id2, weight='weight')
        d_lin_g = round(float(sum(G_sim_lin[p_sim[m]][p_sim[m+1]]['weight'] for m in range(len(p_sim)-1))), 4)
    else:
        d_lin_g = np.nan
        
    case_results.append({
        'Pair': f"{t1} <-> {t2}",
        'Category': rel_desc,
        'Graph Sim (1D)': scores.get('graph_sim_1d', np.nan),
        '2D Spatial Sim': w2,
        'Lin Sim (IC)': scores.get('lin_sim', np.nan),
        'WUP Sim (Depth)': scores.get('wup_depth_sim', np.nan),
        'Jaccard IC Sim': scores.get('jaccard_ic_sim', np.nan),
        'Jaccard Anc Sim': scores.get('jaccard_anc_sim', np.nan),
        'Resnik Norm Sim': scores.get('resnik_norm_sim', np.nan),
        'GraphIC Sim (Blocked)': scores.get('graphic_sim_b', np.nan),
        'GraphIC Sim (Unblocked)': scores.get('graphic_sim_u', np.nan),
        'MICA Label': scores.get('mica_label', 'N/A')
    })

df_case_benchmark = pd.DataFrame(case_results)
print("=== Benchmark Biological Case Studies: All Metrics Side-by-Side ===")
print(df_case_benchmark[['Pair', 'Category', 'Graph Sim (1D)', '2D Spatial Sim', 'Lin Sim (IC)', 'WUP Sim (Depth)', 'Jaccard IC Sim', 'Resnik Norm Sim', 'GraphIC Sim (Blocked)', 'GraphIC Sim (Unblocked)', 'MICA Label']].to_string(index=False))


=== Benchmark Biological Case Studies: All Metrics Side-by-Side ===
                         Pair                        Category  Graph Sim (1D)  2D Spatial Sim  Lin Sim (IC)  WUP Sim (Depth)  Jaccard IC Sim  Resnik Norm Sim  GraphIC Sim (Blocked)  GraphIC Sim (Unblocked)              MICA Label
    saliva <-> salivary gland               Secretome / Fluid          0.7503          0.7766        0.9987           1.0000          0.4267           0.9974                 0.3148                   0.2785                  saliva
  kidney <-> cortex of kidney                   Organ Subpart          0.6593          0.6774        0.8925           1.0000          0.9461           0.8058                 1.0000                   1.0000  mesonephric epithelium
       blood plasma <-> blood                  Fluid Fraction          0.7549          0.6627        1.0000           1.0000          1.0000           1.0000                 1.0000                   1.0000            blood plasma
brain <-> ce

In [9]:
# 7. Corpus-Wide Multi-Metric Evaluation Across All 4,676 Overlapping Runs
def evaluate_all_models_on_run(row):
    if row['is_agree']:
        t_norm = resolve_term(str(row['tissue']).lower().strip())
        return (0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 'Exact Match')
        
    t_agent = str(row['tissue']).lower().strip()
    t_mlm = str(row['tissue_mlm']).lower().strip()
    t_agent_parts = [p.strip() for p in t_agent.split(';') if p.strip()]
    
    if t_mlm in t_agent_parts:
        t_norm = resolve_term(t_mlm)
        return (0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 'Exact Match')
        
    id_mlm = resolve_term(t_mlm)
    p_mlm = term_coords_2d.get(t_mlm)
    if not id_mlm:
        return (np.nan, 0.0, np.nan, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 'Unresolved')
        
    best_d1 = None
    best_d2 = None
    best_lin_s = None
    best_wup_d_s = None
    best_j_ic = None
    best_j_anc = None
    best_resnik = None
    best_g_b = None
    best_g_u = None
    
    for part in t_agent_parts:
        id_agent = resolve_term(part)
        p_agent = term_coords_2d.get(part)
        if id_agent:
            if id_agent == id_mlm:
                return (0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 'Exact Match')
                
            scores = compute_all_metrics_pairwise(id_agent, id_mlm)
            
            d1 = scores['graph_dist_1d']
            if not np.isinf(d1):
                if best_d1 is None or d1 < best_d1:
                    best_d1 = d1
                    
            if p_agent is not None and p_mlm is not None:
                d2 = float(np.linalg.norm(p_agent - p_mlm))
                if best_d2 is None or d2 < best_d2:
                    best_d2 = d2
                    
            if best_lin_s is None or scores['lin_sim'] > best_lin_s:
                best_lin_s = scores['lin_sim']
                best_wup_d_s = scores['wup_depth_sim']
                best_j_ic = scores['jaccard_ic_sim']
                best_j_anc = scores['jaccard_anc_sim']
                best_resnik = scores['resnik_norm_sim']
                best_g_b = scores['graphic_sim_b']
                best_g_u = scores['graphic_sim_u']
                
    w1 = round(1.0 / (1.0 + best_d1), 4) if best_d1 is not None else 0.0
    w2 = round(1.0 / (1.0 + best_d2), 4) if best_d2 is not None else 0.0
    
    return (
        round(best_d1, 4) if best_d1 is not None else np.nan, w1,
        round(best_d2, 4) if best_d2 is not None else np.nan, w2,
        best_lin_s if best_lin_s is not None else 0.0, round(1.0 - best_lin_s, 4) if best_lin_s is not None else 1.0,
        best_wup_d_s if best_wup_d_s is not None else 0.0, round(1.0 - best_wup_d_s, 4) if best_wup_d_s is not None else 1.0,
        best_j_ic if best_j_ic is not None else 0.0, round(1.0 - best_j_ic, 4) if best_j_ic is not None else 1.0,
        best_j_anc if best_j_anc is not None else 0.0,
        best_resnik if best_resnik is not None else 0.0,
        best_g_b if best_g_b is not None else 0.0,
        best_g_u if best_g_u is not None else 0.0,
        'Non-Exact'
    )

print("Evaluating all metrics across 4,676 overlapping runs...")
run_all_eval = df_filtered.apply(evaluate_all_models_on_run, axis=1)

df_filtered['graph_dist_1d'] = [r[0] for r in run_all_eval]
df_filtered['graph_sim_1d'] = [r[1] for r in run_all_eval]
df_filtered['spatial_dist_2d'] = [r[2] for r in run_all_eval]
df_filtered['spatial_sim_2d'] = [r[3] for r in run_all_eval]
df_filtered['lin_sim'] = [r[4] for r in run_all_eval]
df_filtered['lin_dist'] = [r[5] for r in run_all_eval]
df_filtered['wup_depth_sim'] = [r[6] for r in run_all_eval]
df_filtered['wup_depth_dist'] = [r[7] for r in run_all_eval]
df_filtered['jaccard_ic_sim'] = [r[8] for r in run_all_eval]
df_filtered['jaccard_ic_dist'] = [r[9] for r in run_all_eval]
df_filtered['jaccard_anc_sim'] = [r[10] for r in run_all_eval]
df_filtered['resnik_norm_sim'] = [r[11] for r in run_all_eval]
df_filtered['graphic_sim_b'] = [r[12] for r in run_all_eval]
df_filtered['graphic_sim_u'] = [r[13] for r in run_all_eval]

# Concordance Tiers Comparison
def get_tier(score, th_near=0.70, th_dist=0.30):
    if score == 1.0:
        return 'Exact Match'
    elif score >= th_near:
        return 'Near-Agreement'
    elif score >= th_dist:
        return 'Distant-Agreement'
    else:
        return 'Disagreement'

df_filtered['tier_lin'] = df_filtered['lin_sim'].apply(lambda s: get_tier(s, 0.70, 0.30))
df_filtered['tier_wup'] = df_filtered['wup_depth_sim'].apply(lambda s: get_tier(s, 0.75, 0.50))
df_filtered['tier_jaccard_ic'] = df_filtered['jaccard_ic_sim'].apply(lambda s: get_tier(s, 0.30, 0.08))
df_filtered['tier_graphic_b'] = df_filtered['graphic_sim_b'].apply(lambda s: get_tier(s, 0.70, 0.30))
df_filtered['tier_graphic_u'] = df_filtered['graphic_sim_u'].apply(lambda s: get_tier(s, 0.70, 0.30))

print("\n=== Concordance Tier Summary Across Evaluated Metrics ===")
tier_comp = pd.DataFrame({
    '1D Graph Sim': df_filtered['graph_sim_1d'].apply(lambda s: get_tier(s, 0.55, 0.40)).value_counts(normalize=True)*100,
    '2D Spatial Sim': df_filtered['spatial_sim_2d'].apply(lambda s: get_tier(s, 0.65, 0.45)).value_counts(normalize=True)*100,
    'Lin Sim (IC)': df_filtered['tier_lin'].value_counts(normalize=True)*100,
    'WUP Sim (Depth)': df_filtered['tier_wup'].value_counts(normalize=True)*100,
    'Jaccard IC Sim': df_filtered['tier_jaccard_ic'].value_counts(normalize=True)*100,
    'GraphIC Sim (Blocked)': df_filtered['tier_graphic_b'].value_counts(normalize=True)*100,
    'GraphIC Sim (Unblocked)': df_filtered['tier_graphic_u'].value_counts(normalize=True)*100,
}).round(2)

print(tier_comp.to_string())


Evaluating all metrics across 4,676 overlapping runs...

=== Concordance Tier Summary Across Evaluated Metrics ===
                   1D Graph Sim  2D Spatial Sim  Lin Sim (IC)  WUP Sim (Depth)  Jaccard IC Sim  GraphIC Sim (Blocked)  GraphIC Sim (Unblocked)
Disagreement               5.35            8.79          7.06             0.86            8.75                  20.53                    21.56
Distant-Agreement         14.16            7.14          8.81            12.92           11.91                   1.15                     0.11
Exact Match               77.07           77.07         77.07            80.47           77.07                  78.27                    78.27
Near-Agreement             3.42            6.99          7.06             5.75            2.27                   0.04                     0.06


In [10]:
# 8. Comprehensive Comparison Across ALL 133 Unique Pairs in Corpus
pair_counts = df_filtered.groupby(['tissue', 'tissue_mlm']).size().reset_index(name='run_count').sort_values(by='run_count', ascending=False)

all_pairs_comprehensive = []
for idx, row in pair_counts.iterrows():
    t_a = row['tissue']
    t_b = row['tissue_mlm']
    count = row['run_count']
    
    t_a_parts = [p.strip() for p in str(t_a).lower().split(';') if p.strip()]
    t_b_norm = str(t_b).lower().strip()
    id_b = resolve_term(t_b_norm)
    p_b = term_coords_2d.get(t_b_norm)
    
    best_d1 = None
    best_d2 = None
    best_scores = {}
    
    for part in t_a_parts:
        id_a = resolve_term(part)
        p_a = term_coords_2d.get(part)
        if id_a and id_b:
            if id_a == id_b:
                best_d1 = 0.0
                best_d2 = 0.0
                best_scores = compute_all_metrics_pairwise(id_a, id_b)
                break
                
            scores = compute_all_metrics_pairwise(id_a, id_b)
            d1 = scores['graph_dist_1d']
            if not np.isinf(d1):
                if best_d1 is None or d1 < best_d1:
                    best_d1 = d1
                    
            if p_a is not None and p_b is not None:
                d2 = float(np.linalg.norm(p_a - p_b))
                if best_d2 is None or d2 < best_d2:
                    best_d2 = d2
                    
            if not best_scores or scores['lin_sim'] > best_scores.get('lin_sim', -1):
                best_scores = scores
                
    w1 = round(1.0 / (1.0 + best_d1), 4) if best_d1 is not None else 0.0
    w2 = round(1.0 / (1.0 + best_d2), 4) if best_d2 is not None else 0.0
    lin_s = best_scores.get('lin_sim', 0.0)
    
    tier = 'Exact Match' if best_d1 == 0.0 else ('Near-Agreement' if lin_s >= 0.70 else ('Distant-Agreement' if lin_s >= 0.30 else 'Disagreement'))
    
    all_pairs_comprehensive.append({
        'HAMLET (Agent)': t_a,
        'MLMarker': t_b,
        'Run Count': count,
        'Tier': tier,
        '1D Graph Dist': round(best_d1, 4) if best_d1 is not None else np.nan,
        '1D Graph Sim': w1,
        '2D Spatial Dist': round(best_d2, 4) if best_d2 is not None else np.nan,
        '2D Spatial Sim': w2,
        'Lin Sim (IC)': lin_s,
        'Lin Dist (1-S)': round(1.0 - lin_s, 4),
        'WuPalmer Sim (Depth)': best_scores.get('wup_depth_sim', np.nan),
        'Jaccard IC Sim': best_scores.get('jaccard_ic_sim', np.nan),
        'Jaccard Anc Sim': best_scores.get('jaccard_anc_sim', np.nan),
        'Resnik Norm Sim': best_scores.get('resnik_norm_sim', np.nan),
        'GraphIC Sim (Blocked)': best_scores.get('graphic_sim_b', np.nan),
        'GraphIC Sim (Unblocked)': best_scores.get('graphic_sim_u', np.nan),
        'MICA Label': best_scores.get('mica_label', 'N/A')
    })

df_all_comp = pd.DataFrame(all_pairs_comprehensive)
df_all_comp.to_csv(os.path.join(PATH_OUTPUT, "all_metrics_comprehensive_comparison.csv"), index=False)

print(f"Exported all {len(df_all_comp)} pairs to Output/all_metrics_comprehensive_comparison.csv\n")
print("=== Top 20 Most Frequent Pairs Across All Evaluated Metrics ===")
cols_disp = ['HAMLET (Agent)', 'MLMarker', 'Run Count', 'Tier', '1D Graph Sim', '2D Spatial Sim', 'Lin Sim (IC)', 'WuPalmer Sim (Depth)', 'Jaccard IC Sim', 'GraphIC Sim (Blocked)', 'GraphIC Sim (Unblocked)', 'MICA Label']
print(df_all_comp.head(20)[cols_disp].to_string(index=False))


Exported all 133 pairs to Output/all_metrics_comprehensive_comparison.csv

=== Top 20 Most Frequent Pairs Across All Evaluated Metrics ===
                                                                   HAMLET (Agent)        MLMarker  Run Count              Tier  1D Graph Sim  2D Spatial Sim  Lin Sim (IC)  WuPalmer Sim (Depth)  Jaccard IC Sim  GraphIC Sim (Blocked)  GraphIC Sim (Unblocked)                      MICA Label
                                                                            brain           Brain       1568       Exact Match        1.0000          1.0000        1.0000                1.0000          1.0000                 1.0000                   1.0000                           brain
                                                                            liver           Liver        569       Exact Match        1.0000          1.0000        1.0000                1.0000          1.0000                 1.0000                   1.0000                           

In [11]:
# 9. Publication Figure: Discrimination Power & Metric Correlation Benchmark
fig, axes = plt.subplots(1, 3, figsize=(26, 7.5))

# --- Panel 1: Pairwise Metric Correlation Heatmap ---
metric_cols = ['1D Graph Sim', '2D Spatial Sim', 'Lin Sim (IC)', 'WuPalmer Sim (Depth)', 'Jaccard IC Sim', 'Jaccard Anc Sim', 'Resnik Norm Sim', 'GraphIC Sim (Blocked)', 'GraphIC Sim (Unblocked)']
df_corr_matrix = df_all_comp[metric_cols].corr(method='spearman')

sns.heatmap(df_corr_matrix, cmap='coolwarm', annot=True, fmt='.3f', vmin=0.4, vmax=1.0,
            linewidths=0.8, linecolor='white', cbar_kws={'label': 'Spearman Rank Correlation ($\\rho$)'}, ax=axes[0])
axes[0].set_title('A. Cross-Metric Spearman Rank Correlation ($\\rho$)', fontsize=12.5, weight='bold', pad=15)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=35, ha='right', fontsize=9, weight='bold')
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0, fontsize=9, weight='bold')

# --- Panel 2: Biological Category Separation (Contrast Power) ---
# Group pairs into benchmark ground-truth categories
def categorize_pair(row):
    if row['Tier'] == 'Exact Match':
        return 'Exact Match'
    t_a = str(row['HAMLET (Agent)']).lower()
    t_b = str(row['MLMarker']).lower()
    if any(k in t_a for k in ['saliva', 'plasma', 'serum', 'urine', 'semen', 'fluid', 'csf', 'bal']):
        return 'Secretome / Fluids'
    elif any(k in t_a for k in ['cortex', 'duodenum', 'rectum', 'appendix']):
        return 'Subparts / Subdivisions'
    elif any(k in t_a for k in ['monocyte', 'b-cell', 't cell', 'pbmc', 'neutrophil', 'stem']):
        return 'Cellular Lineages'
    elif row['Tier'] == 'Disagreement':
        return 'True Conflicts'
    else:
        return 'Same System / Distant'

df_all_comp['BioCategory'] = df_all_comp.apply(categorize_pair, axis=1)
category_order = ['Exact Match', 'Subparts / Subdivisions', 'Secretome / Fluids', 'Cellular Lineages', 'Same System / Distant', 'True Conflicts']

# Calculate mean score per category for each metric
cat_means = df_all_comp.groupby('BioCategory')[['Lin Sim (IC)', '1D Graph Sim', '2D Spatial Sim', 'WuPalmer Sim (Depth)', 'Jaccard IC Sim', 'GraphIC Sim (Blocked)', 'GraphIC Sim (Unblocked)']].mean().reindex(category_order)

cat_means.plot(kind='bar', ax=axes[1], colormap='viridis', width=0.8, edgecolor='black', linewidth=0.8)
axes[1].set_title('B. Metric Discrimination Across Biological Relationship Types', fontsize=12.5, weight='bold', pad=15)
axes[1].set_ylabel('Mean Similarity Score ($[0, 1]$)', fontsize=11, weight='bold')
axes[1].set_xlabel('Biological Relationship Category', fontsize=11, weight='bold')
axes[1].set_xticklabels(category_order, rotation=35, ha='right', fontsize=9, weight='bold')
axes[1].legend(title='Metric', fontsize=9, title_fontsize=10, loc='upper right')
axes[1].grid(axis='y', alpha=0.5)

# --- Panel 3: Contrast Ratio (Separability Index) ---
# Contrast = Score(Secretome + Subpart) - Score(True Conflict)
contrast_data = []
for col in ['Lin Sim (IC)', '1D Graph Sim', '2D Spatial Sim', 'WuPalmer Sim (Depth)', 'Jaccard IC Sim', 'Resnik Norm Sim', 'GraphIC Sim (Blocked)', 'GraphIC Sim (Unblocked)']:
    near_score = df_all_comp[df_all_comp['BioCategory'].isin(['Subparts / Subdivisions', 'Secretome / Fluids'])][col].mean()
    conflict_score = df_all_comp[df_all_comp['BioCategory'] == 'True Conflicts'][col].mean()
    contrast_ratio = near_score - conflict_score
    contrast_data.append({'Metric': col, 'Near Score': near_score, 'Conflict Score': conflict_score, 'Contrast (Delta)': contrast_ratio})

df_contrast = pd.DataFrame(contrast_data).sort_values(by='Contrast (Delta)', ascending=True)

colors_contrast = plt.cm.plasma(np.linspace(0.2, 0.85, len(df_contrast)))
bars = axes[2].barh(df_contrast['Metric'], df_contrast['Contrast (Delta)'], color=colors_contrast, edgecolor='black', linewidth=0.8)
axes[2].set_title('C. Separability Index (Contrast $\\Delta = \\text{Near} - \\text{Conflict}$)', fontsize=12.5, weight='bold', pad=15)
axes[2].set_xlabel('Separability $\\Delta$ (Higher is Better)', fontsize=11, weight='bold')
axes[2].grid(axis='x', alpha=0.5)

for i, v in enumerate(df_contrast['Contrast (Delta)']):
    axes[2].text(v + 0.01, i, f"+{v:.3f}", va='center', fontsize=9.5, weight='bold')

plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT, "metric_discrimination_and_correlation_benchmark.png"), dpi=300, bbox_inches='tight')
plt.close()

print("Discrimination & Correlation benchmark figure saved to Output/metric_discrimination_and_correlation_benchmark.png")
print("\n=== Metric Separability & Contrast Scorecard ===")
print(df_contrast.to_string(index=False))


Discrimination & Correlation benchmark figure saved to Output/metric_discrimination_and_correlation_benchmark.png

=== Metric Separability & Contrast Scorecard ===
                 Metric  Near Score  Conflict Score  Contrast (Delta)
           1D Graph Sim    0.467068         0.42328          0.043788
         2D Spatial Sim    0.593864         0.48768          0.106184
   WuPalmer Sim (Depth)    0.712673         0.59018          0.122493
GraphIC Sim (Unblocked)    0.222465         0.07339          0.149075
  GraphIC Sim (Blocked)    0.225419         0.06054          0.164879
         Jaccard IC Sim    0.210981         0.02665          0.184331
           Lin Sim (IC)    0.483732         0.19010          0.293632
        Resnik Norm Sim    0.446527         0.14343          0.303097


In [12]:
# 10. Complete Performance Recap & Summary Tables Across All Evaluated Metrics
print("=" * 115)
print("              COMPREHENSIVE PERFORMANCE RECAP & BENCHMARK SUMMARY")
print("=" * 115)

# --- 1. Biological Category Performance Breakdown (Mean +/- Std) ---
cat_summary = []
for cat in category_order:
    sub_df = df_all_comp[df_all_comp['BioCategory'] == cat]
    n_pairs = len(sub_df)
    n_runs = sub_df['Run Count'].sum()
    cat_summary.append({
        'Biological Category': cat,
        'Pairs': n_pairs,
        'Total Runs': n_runs,
        'Lin Sim (IC)': f"{sub_df['Lin Sim (IC)'].mean():.4f} +/- {sub_df['Lin Sim (IC)'].std():.3f}" if n_pairs > 1 else f"{sub_df['Lin Sim (IC)'].mean():.4f}",
        'Resnik Norm': f"{sub_df['Resnik Norm Sim'].mean():.4f} +/- {sub_df['Resnik Norm Sim'].std():.3f}" if n_pairs > 1 else f"{sub_df['Resnik Norm Sim'].mean():.4f}",
        '1D Graph Sim': f"{sub_df['1D Graph Sim'].mean():.4f} +/- {sub_df['1D Graph Sim'].std():.3f}" if n_pairs > 1 else f"{sub_df['1D Graph Sim'].mean():.4f}",
        '2D Spatial Sim': f"{sub_df['2D Spatial Sim'].mean():.4f} +/- {sub_df['2D Spatial Sim'].std():.3f}" if n_pairs > 1 else f"{sub_df['2D Spatial Sim'].mean():.4f}",
        'WuPalmer (Depth)': f"{sub_df['WuPalmer Sim (Depth)'].mean():.4f} +/- {sub_df['WuPalmer Sim (Depth)'].std():.3f}" if n_pairs > 1 else f"{sub_df['WuPalmer Sim (Depth)'].mean():.4f}",
        'Jaccard IC': f"{sub_df['Jaccard IC Sim'].mean():.4f} +/- {sub_df['Jaccard IC Sim'].std():.3f}" if n_pairs > 1 else f"{sub_df['Jaccard IC Sim'].mean():.4f}",
        'GraphIC (Blocked)': f"{sub_df['GraphIC Sim (Blocked)'].mean():.4f} +/- {sub_df['GraphIC Sim (Blocked)'].std():.3f}" if n_pairs > 1 else f"{sub_df['GraphIC Sim (Blocked)'].mean():.4f}",
        'GraphIC (Unblocked)': f"{sub_df['GraphIC Sim (Unblocked)'].mean():.4f} +/- {sub_df['GraphIC Sim (Unblocked)'].std():.3f}" if n_pairs > 1 else f"{sub_df['GraphIC Sim (Unblocked)'].mean():.4f}"
    })

df_cat_recap = pd.DataFrame(cat_summary)
print("\n[TABLE 1] Performance Breakdown Across Biological Relationship Categories:")
print(df_cat_recap.to_string(index=False))

# --- 2. Separability Index & Contrast Ranking ---
scorecard_rows = []
for col, name in [
    ('Lin Sim (IC)', 'Lin IC Similarity'),
    ('Resnik Norm Sim', 'Resnik Normalized'),
    ('Jaccard IC Sim', 'Jaccard IC (Weighted)'),
    ('WuPalmer Sim (Depth)', 'Wu & Palmer (Depth)'),
    ('2D Spatial Sim', '2D Spatial MDS Sim'),
    ('1D Graph Sim', '1D Graph Shortest Sim'),
    ('GraphIC Sim (Blocked)', 'GraphIC Sim (Blocked)'),
    ('GraphIC Sim (Unblocked)', 'GraphIC Sim (Unblocked)')
]:
    near_val = df_all_comp[df_all_comp['BioCategory'].isin(['Subparts / Subdivisions', 'Secretome / Fluids'])][col].mean()
    conflict_val = df_all_comp[df_all_comp['BioCategory'] == 'True Conflicts'][col].mean()
    delta = near_val - conflict_val
    ratio = near_val / conflict_val if conflict_val > 0 else np.inf
    scorecard_rows.append({
        'Metric Name': name,
        'Near-Agreement Mean': round(near_val, 4),
        'True Conflict Mean': round(conflict_val, 4),
        'Separability Delta (Near - Conflict)': round(delta, 4),
        'Contrast Ratio (Near / Conflict)': round(ratio, 2)
    })

df_scorecard = pd.DataFrame(scorecard_rows).sort_values(by='Separability Delta (Near - Conflict)', ascending=False)
df_scorecard['Rank'] = range(1, len(df_scorecard) + 1)
cols_sc = ['Rank', 'Metric Name', 'Near-Agreement Mean', 'True Conflict Mean', 'Separability Delta (Near - Conflict)', 'Contrast Ratio (Near / Conflict)']

print("\n" + "=" * 115)
print("[TABLE 2] Quantitative Separability & Contrast Scorecard (Ranked):")
print(df_scorecard[cols_sc].to_string(index=False))

# --- 3. Corpus Concordance Rate Summary (4,676 Runs) ---
print("\n" + "=" * 115)
print("[TABLE 3] Corpus-Wide Concordance Distribution (4,676 Total Overlapping Runs):")
df_conc_recap = pd.DataFrame({
    '1D Graph Sim': df_filtered['graph_sim_1d'].apply(lambda s: get_tier(s, 0.55, 0.40)).value_counts(normalize=True)*100,
    '2D Spatial Sim': df_filtered['spatial_sim_2d'].apply(lambda s: get_tier(s, 0.65, 0.45)).value_counts(normalize=True)*100,
    'Lin Sim (IC)': df_filtered['tier_lin'].value_counts(normalize=True)*100,
    'WuPalmer (Depth)': df_filtered['tier_wup'].value_counts(normalize=True)*100,
    'Jaccard IC Sim': df_filtered['tier_jaccard_ic'].value_counts(normalize=True)*100,
    'GraphIC Sim (Blocked)': df_filtered['tier_graphic_b'].value_counts(normalize=True)*100,
    'GraphIC Sim (Unblocked)': df_filtered['tier_graphic_u'].value_counts(normalize=True)*100,
}).fillna(0).round(2).reindex(['Exact Match', 'Near-Agreement', 'Distant-Agreement', 'Disagreement'])

df_conc_recap.loc['Effective Concordance (Exact + Near)'] = (
    df_conc_recap.loc['Exact Match'] + df_conc_recap.loc['Near-Agreement']
)
print(df_conc_recap.to_string())

# --- 4. Direct Benchmark Ground-Truth Pairs Comparison ---
print("\n" + "=" * 115)
print("[TABLE 4] Direct Comparison on Key Biological Ground-Truth Pairs:")
cols_bm = ['Pair', 'Category', 'Graph Sim (1D)', '2D Spatial Sim', 'Lin Sim (IC)', 'WUP Sim (Depth)', 'Jaccard IC Sim', 'Resnik Norm Sim', 'GraphIC Sim (Blocked)', 'GraphIC Sim (Unblocked)', 'MICA Label']
print(df_case_benchmark[cols_bm].to_string(index=False))
print("=" * 115)



              COMPREHENSIVE PERFORMANCE RECAP & BENCHMARK SUMMARY

[TABLE 1] Performance Breakdown Across Biological Relationship Categories:
    Biological Category  Pairs  Total Runs     Lin Sim (IC)      Resnik Norm     1D Graph Sim   2D Spatial Sim WuPalmer (Depth)       Jaccard IC GraphIC (Blocked) GraphIC (Unblocked)
            Exact Match     60        3604 1.0000 +/- 0.000 1.0000 +/- 0.000 1.0000 +/- 0.000 1.0000 +/- 0.000 1.0000 +/- 0.000 1.0000 +/- 0.000  1.0000 +/- 0.000    1.0000 +/- 0.000
Subparts / Subdivisions      3          43 0.8161 +/- 0.204 0.7208 +/- 0.275 0.5862 +/- 0.066 0.7062 +/- 0.189 0.8070 +/- 0.334 0.5036 +/- 0.402  0.5847 +/- 0.487    0.6182 +/- 0.437
     Secretome / Fluids     25         413 0.4438 +/- 0.278 0.4107 +/- 0.247 0.4528 +/- 0.176 0.5804 +/- 0.250 0.7004 +/- 0.191 0.1728 +/- 0.209  0.1786 +/- 0.277    0.1709 +/- 0.275
      Cellular Lineages      8          38 0.5508 +/- 0.190 0.4611 +/- 0.219 0.4339 +/- 0.054 0.4567 +/- 0.077 0.6167 +/- 0.21

## 11. Final Quantitative Scorecard & Strategic Recommendation

### Comprehensive Comparison Matrix:
#### Do not take into account stars rating

| Evaluation Criterion | 1. Lin IC Similarity ($\text{Lin}$) | 2. Resnik Normalized | 3. GraphIC (Blocked) | 4. GraphIC (Unblocked) | 5. Jaccard IC ($\text{IC}$) | 6. 1D Graph Distance ($\text{graph}$) | 7. 2D Metric MDS ($\text{2D}$) | 8. Wu & Palmer (Depth) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **Secretome Resolution** (*Saliva, Plasma, Urine*) | **★★★★★** ( = 0.86-1.00$) | **★★★★★** ( = 0.80-1.00$) | **★★☆☆☆** ( = 0.06-1.00$) | **★★☆☆☆** ( = 0.04-1.00$) | **★★★☆☆** ( = 0.19-1.00$) | **★★★★☆** ( = 0.61-0.75$) | **★★★★☆** ( = 0.66-0.78$) | **★★☆☆☆** (Over-simplifies) |
| **Subpart Resolution** (*Cortex, Duodenum*) | **★★★★★** ( = 0.89-0.92$) | **★★★★☆** ( = 0.81-0.86$) | **★★☆☆☆** ( = 0.09-1.00$) | **★★☆☆☆** ( = 0.13-1.00$) | **★★☆☆☆** ( = 0.08-0.95$) | **★★★★☆** ( = 0.66-0.76$) | **★★★★★** ( = 0.68-0.87$) | **★★★★☆** ( = 0.70-1.00$) |
| **Cellular Lineages** (*Monocytes $\to$ Blood*) | **★★★★★** ( = 0.831$) | **★★★★☆** ( = 0.802$) | **★☆☆☆☆** ( = 0.064$) | **★☆☆☆☆** ( = 0.108$) | **★★☆☆☆** ( = 0.188$) | **★★★☆☆** ( = 0.426$) | **★★★★☆** ( = 0.513$) | **★★★★☆** ( = 0.818$) |
| **Conflict Rejection** (*Skin $\leftrightarrow$ Ovary*) | **★★★★★** ( = 0.136$) | **★★★★★** ( = 0.094$) | **★★★★★** ( = 0.030$) | **★★★★★** ( = 0.069$) | **★★★★★** ( = 0.026$) | **★★★★☆** ( = 0.386$) | **★★★★☆** ( = 0.338$) | **★☆☆☆☆** ( = 0.571$, Bad!) |
| **Separability Index $\Delta$** | **+0.294 (Rank 2)** | **+0.303 (Rank 1)** | **+0.165 (Rank 4)** | **+0.149 (Rank 5)** | **+0.184 (Rank 3)** | **+0.044 (Rank 8)** | **+0.106 (Rank 7)** | **+0.122 (Rank 6)** |
| **Mathematical Bounds** | Strictly $[0, 1]$ | Strictly $[0, 1]$ | Strictly $[0, 1]$ | Strictly $[0, 1]$ | Strictly $[0, 1]$ | Continuous $[0, 1]$ | Continuous $[0, 1]$ | Strictly $[0, 1]$ |
| **Dynamic Range** | Excellent (.0 - 1.0$) | Excellent (.0 - 1.0$) | Compressed (.0 - 1.0$) | Compressed (.0 - 1.0$) | Skewed near zero | Compressed (.30 - 1.0$) | Compressed (.35 - 1.0$) | Truncated at root (.57$) |

---

### Strategic Recommendation for Paper & Downstream Analysis:

1. **Primary Recommended Similarity Metric: Resnik Normalized / Lin IC Similarity (${\text{Lin}}$)**
   - **Why**: Possesses the **highest biological Separability Index ($\Delta = 0.30$ and .29$)**. They award near-perfect scores ( \ge 0.85$) to biological secretome transitions (*Saliva $\to$ Salivary Gland*, *Blood Plasma $\to$ Blood*, *Seminal Plasma $\to$ Prostate*) and subparts (*Duodenum $\to$ Small Intestine*, *Kidney Cortex $\to$ Kidney*), while aggressively punishing false cross-organ predictions ( < 0.15$ for *Skin $\to$ Ovary*,  < 0.25$ for *Brain $\to$ Heart*).
2. **Complementary Topological Metric: 1D Graph Distance (${\text{graph}}$)**
   - **Why**: Provides step-by-step biological explainability and path provenance along explicit UBERON/CL relations.
3. **Metric to AVOID: Standard Wu-Palmer (Depth) and GraphIC (GO-universal)**
   - **Why Wu-Palmer**: Suffers from **depth-inflation** at upper taxonomy levels, giving an unacceptably high similarity ( = 0.5714$) to completely unrelated organs sharing only distant roots.
   - **Why GraphIC**: Suffers from extreme score compression and instability. It heavily punishes specific biological subparts and neighbors due to multiplicative compounding from deep ontology hierarchies (e.g. Brain $\to$ Cerebral Cortex  < 0.05).
4. **To explore: Use the Human Protein Atlas to perform similarity scoring**  
   - Could add some bias? MLMarker is also a tool that compare proteomes to assign a tissue prediction to a sample.
